In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("source_table_name", "")
dbutils.widgets.text("source_file_format","")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_system = dbutils.widgets.get("source_system")
source_table_name = dbutils.widgets.get("source_table_name")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
base_landing_path = "/Volumes/databeli_adb/landing/landing_volume"
source_path = f"{base_landing_path}/{source_system}"
table_path = f"{source_path}/{source_table_name}"
landing_path = f"{table_path}/{landing_timestamp}"

def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False
    
missing_path = next((path for path in [source_path, table_path, landing_path] if not path_exists(path)), None)

if missing_path:
    message = (f"No new files are received for source_system: {source_system}, "
               f"table_name: {source_table_name}, landing_timestamp: {landing_timestamp}"
               )
    print(message)
    dbutils.notebook.exit(message)

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv")\
                            .option("header", "true")\
                            .option("inferSchema", "true")\
                            .load(f"/Volumes/databeli_adb/landing/landing_volume/{source_system}/{source_table_name}/{landing_timestamp}/")

else:
    landing_df = spark.read.format("parquet")\
                        .load(f"/Volumes/databeli_adb/landing/landing_volume/{source_system}/{source_table_name}/{landing_timestamp}/")

In [0]:
from pyspark.sql.functions import *
updated_df = landing_df.withColumn('landing_timestamp', lit(landing_timestamp))\
                        .withColumn('insert_timestamp', current_timestamp())

In [0]:
updated_df.write.mode("append")\
                .saveAsTable(f"databeli_adb.bronze.{source_table_name}")